<a href="https://colab.research.google.com/github/kanebako-s/RealEstate_analysis/blob/main/RealEstate_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# --- ライブラリのインポート　---
import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from pathlib import Path

# --- データの表示設定 ---
pd.options.display.max_columns = None # 列数が多い時、省略（...）せずに全部表示する
pd.set_option('display.float_format', '{:.2f}'.format) # 小数点以下を2桁に固定して見やすくする

# --- 日本語化（重要！） ---
!pip install japanize-matplotlib
import japanize_matplotlib # これでグラフの日本語が「□」にならずに済む

# --- Worldcloudのインストール ---
!pip install wordcloud
from wordcloud import WordCloud

分析の土台となるファイルの読み込み :

In [19]:
# パスを定義
PROJECT_ROOT = Path("/content/drive/MyDrive/RealEstate_DateAnalist_Learning")
GOV_DATA_BASE = PROJECT_ROOT / "hyokasyo-2026-東京都 2"
KAGGLE_DATA_BASE = PROJECT_ROOT / "archive-2"

# ファイルの読み込み
df_gov = pd.read_csv(
    GOV_DATA_BASE / "2026_TAKUCHI_k_13.csv",
    encoding='cp932',
    header=None,
    dtype=str  # 全て文字列で読み込む安全策
)
df_kaggle = pd.read_csv(
    KAGGLE_DATA_BASE / "Kaggle_All_prefectures_buildings_with_migration.csv",
    dtype=str  # 全て文字列で読み込む安全策
)

本ステップでは、1400項目を超える国土交通省の地価公示データから、分析の目的に合致する主要な5項目を選出した。公式のレイアウト定義書（PDF）が不在の状況下であったが、以下の特定プロセスを経て列の情報を確定させた。

1. 「千代田区」「〜町〜番地」といった既知の地名や住所が記されていることから 列3と列26は「市区町村」「住所」と推測しました。

2. 住所からその土地の平米単価をgoogle検索をして、その数値と近似している列を特定し、列19が「平米単価」と推測しました。

3. 住所それぞれの最寄駅を調べ一致している列を特定した。したがって列49は「最寄駅」と推測しました。

4. 住所と最寄り駅の距離を一件づつ調べ、その数値と各列の持つ数値を照らし合わせました。そして「駅まで何mか」という情報を持つ列を列50と推測しました

In [20]:
# 分析に使用する列を選出
df_selected = df_gov.iloc[:, [3, 19, 26, 49, 50]].copy()

# 抽出した列に名前をつける（日本語でOK、後のグラフ作成が楽になります）
df_selected.columns = ['市区町村', '公示地価', '住所', '最寄り駅', '駅距離']

# 公示地価と駅距離を数値型に変換
# errors='coerce' をつけることで、万が一数値以外の文字が入っていてもエラーにならず「欠損値(NaN)」として処理してくれます
df_selected['公示地価'] = pd.to_numeric(df_selected['公示地価'], errors='coerce')
df_selected['駅距離'] = pd.to_numeric(df_selected['駅距離'], errors='coerce')

In [21]:
print(df_selected.describe())

             公示地価     駅距離
count     5104.00 5104.00
mean   1483484.48  829.53
std    3847920.76  882.29
min       5400.00    0.00
25%     291750.00  300.00
50%     563000.00  600.00
75%    1180000.00 1100.00
max   67600000.00 9100.00


結果 : 特定した公示地価と駅距離は不動産市場の常識的に妥当である。

In [22]:
# 欠損値の数を確認
print(df_selected.isnull().sum())

市区町村    0
公示地価    0
住所      0
最寄り駅    0
駅距離     0
dtype: int64


洞察 : 政府のデータは情報が一緒くたになっているだけで書式は統一されていてきれいなのかもしれない。

本ステップでは土地の「用途区分コード（住宅地、商業地、工業地、宅地見込地）」と上物がある場合の「築年数」の情報を抽出します。

** 用途区分コードの特定 **

仮説 : 用途区分コードは都市建設法の基づいて割り当てられるため種類数は2〜15種類程度に収まるはずである。

In [23]:
# 1. データの読み込み（列名を強制的に連番にする）
# Why: 行政データはヘッダーがないため、Pandas側で一律に名前を付けるのが最も堅牢です [cite: 340, 365]
df_gov = pd.read_csv('2026_TAKUCHI_k_13.csv', encoding='cp932', header=None)
df_gov.columns = [f"column{str(i).zfill(3)}" for i in range(len(df_gov.columns))]

# 2. DuckDBへの登録
con = duckdb.connect()
con.execute("CREATE OR REPLACE TABLE raw_gov AS SELECT * FROM df_gov")

# 3. 14列目（column013）の抽出実行
# Why: TRIMを使うことで、行政データ特有の末尾の空白を削除し、正確な集計を可能にします [cite: 368]
query = """
SELECT
    TRIM(column013) AS use_code,
    COUNT(*) AS count
FROM raw_gov
GROUP BY use_code
ORDER BY count DESC
"""

res = con.execute(query).df()
print("✅ 用途区分コードの特定に成功しました")
display(res)

FileNotFoundError: [Errno 2] No such file or directory: '2026_TAKUCHI_k_13.csv'

In [ ]:
# 商業地の代表（中央区銀座4丁目：山野楽器付近など）
ginza_check = df_gov[df_gov.iloc[:, 26].str.contains('銀座４丁目', na=False)].iloc[:, [26, 31, 35]]

# 住宅地の代表（千代田区三番町）
sanbancho_check = df_gov[df_gov.iloc[:, 26].str.contains('三番町', na=False)].iloc[:, [26, 31, 35]]

print("--- 銀座（商業地）の列31 ---")
print(ginza_check.head(3))

print("\n--- 三番町（住宅地）の列31 ---")
print(sanbancho_check.head(3))

In [ ]:
df_gov.shape

In [ ]:
df_gov.head()